# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import duckdb      # DuckDB is a database engine that can run SQL queries directly on files
import os          # built-in Python module used to interact with the operating system
import pandas as pd
import numpy as np

con = duckdb.connect()

HF_TOKEN = os.environ["HF_TOKEN"]     # environment variable that contains Hugging Face authentication token

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
  TYPE HUGGINGFACE,
  TOKEN'{HF_TOKEN}'
)
""")

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

query= f"""
SELECT
  client_hash_id AS client_id,
  content_hash_id AS content_id,

  SUM(gsc_impressions) AS gsc_impressions,
  SUM(gsc_clicks) AS gsc_clicks,
  AVG(NULLIF(gsc_avg_position,0)) AS gsc_avg_position,

  SUM(
    CASE
      WHEN ga4_data_available IS TRUE
      THEN ga4_sessions
      ELSE NULL
    END
  ) AS ga4_sessions,

  SUM(
    CASE
      WHEN ga4_data_available IS TRUE
      THEN ga4_engaged_sessions
      ELSE NULL
    END
  ) AS ga4_engaged_sessions,

  BOOL_OR(ga4_data_available) AS ga4_data_available

  FROM read_parquet('{march_path}')
  GROUP BY client_hash_id, content_hash_id
  """
features = con.execute(query).df()

# ----------------------------- Feature engineering ---------------------------------

# 1. Google Search CTR
features["gsc_ctr"] = np.where(
    features["gsc_impressions"] > 0,
    features["gsc_clicks"] / features["gsc_impressions"],
    np.nan
)

# 2. GA4 engagement rate
features["ga4_engagement_rate"] = np.where(
    features["ga4_sessions"] > 0,
    features["ga4_engaged_sessions"] / features["ga4_sessions"],
    np.nan
)

# --------------------------- Missing-value handling ----------------------------------
features["has_ga4_data"] = (
    features["ga4_data_available"].fillna(False).astype(int)
)

numeric_features= [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_ctr",
    "ga4_engagement_rate"
]

for col in numeric_features:
    features[col] = features[col].fillna(features[col].median())

# --------------------------- Categorical handling ----------------------------------
# Convert it to a numeric indicator.
features["ga4_data_available"] = (
    features["ga4_data_available"]
    .fillna(False)
    .astype(int)
)

# Remove IDs from the actual ML feature vector

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_ctr",
    "ga4_engagement_rate",
    "ga4_data_available",
    "has_ga4_data"
]

X = features[feature_columns].copy()

# Categorical handling
print("Feature table shape:", X.shape)
print("\nFeature columns:")
print(X.columns.tolist())

print("\nMissing values after filling:")
print(X.isna().sum())
X.head()

# Verification queries
print("=== DATA QUALITY CHECKS ===\n")

# 1. Grain check (should be one row per content-client)
print("1. Grain verification:")
print(f"   Total rows: {len(features)}")
print(f"   Unique clients: {features['client_id'].nunique()}")
print(f"   Unique contents: {features['content_id'].nunique()}")

# 2. Missing values BEFORE filling
print("\n2. Missing values before filling:")
print(features[['gsc_impressions', 'gsc_clicks', 'ga4_sessions']].isnull().sum())

# 3. Outliers check
print("\n3. Outlier check (max values):")
print(features[['gsc_impressions', 'gsc_clicks', 'ga4_sessions']].describe().loc['max'])

# 4. Distribution check
print("\n4. Feature distributions (after filling):")
print(X.describe())

KeyError: 'HF_TOKEN'

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Answer:**

### gsc_impressions
- **Meaning:** Count of search impressions in March 2026
- **Missing handling:** 0 if no data; aggregated via SUM()
- **Availability:** Observable; historical search data from past

### gsc_clicks  
- **Meaning:** Count of clicks from Google Search results
- **Missing handling:** 0 if no data; aggregated via SUM()
- **Availability:** Historical; knowable before clustering

### gsc_avg_position
- **Meaning:** Average Google Search result position (1 = top result, lower is better)
- **Missing handling:** 0 means "no position data"; NULLIF(position, 0) then AVG() of non-zeros
- **Special note:** Must flag which rows have no position data via has_position_data flag
- **Availability:** Historical search data

### ga4_sessions
- **Meaning:** Count of website sessions from GA4
- **Missing handling:** Only included when ga4_data_available = TRUE; otherwise NULL
- **Availability:** Historical; observed in March 2026

### ga4_engagement_rate
- **Meaning:** Engaged sessions / total sessions (ratio)
- **Missing handling:** Computed only when ga4_sessions > 0; otherwise NaN → median fill
- **Availability:** Historical engagement metric

### ga4_data_available (FLAG)
- **Meaning:** Boolean flag: is GA4 data valid for this row?
- **Why it matters:** When FALSE, ga4 columns are zero-filled but don't represent real zeros
- **Availability:** Metadata flag; critical for interpretation

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Answer:**

I checked the feature set for possible data leakage by making sure that is_declining_label, trend_direction, and trend_pct are not included because they are derived from the outcome or future performance. I also excluded client_id and content_id because they are identifiers rather than useful content characteristics, and I did not use data from future months. The clustering features are restricted to the March 2026 development window, so the feature vector uses only the observed information available in that window.

In [ ]:
# Section 3: Leakage Hunt

leakage_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

feature_columns_used = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_sessions", "ga4_engaged_sessions",
    "gsc_ctr", "ga4_engagement_rate",
    "ga4_data_available", "has_ga4_data"
]

# TEST 1: Check for label leakage
leaked_features = [col for col in leakage_columns if col in feature_columns_used]

print("TEST 1: Checking for label-derived features...")
print(f"Features used: {feature_columns_used}")
print(f"Leakage columns found: {leaked_features}")

if len(leaked_features) == 0:
    print("PASS: No leakage columns detected")
else:
    print("FAIL: Leakage columns found:", leaked_features)

# TEST 2: Verify position = 0 is flagged
print("\nTEST 2: Position data availability")
print(f"Rows in feature set: {len(X)}")
print(f"Missing value check:\n{X.isna().sum()}")

# TEST 3: GA4 flag exists
print("TEST 3: GA4 availability flagging")
print(f"GA4 data available = 1: {(X['ga4_data_available'] == 1).sum()} rows")
print(f"GA4 data available = 0: {(X['ga4_data_available'] == 0).sum()} rows")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
**Answer:**

I excluded `client_id` and `content_id` because they are identifiers and do not describe the characteristics of the content. I excluded `is_declining_label`, `trend_direction`, and `trend_pct` because they are label-derived or related to future performance and could cause leakage. I excluded future-period data because it would not be available at the time of analysis. I also excluded `ga4_data_available` from the clustering features because it indicates whether GA4 data exists rather than describing the content itself.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.